# Equal-intensity simulation: Bonferroni test

This notebook estimates the empirical type-I error of the Bonferroni test under equal persistence intensity. The two populations have the same intensity but different within-diagram dependence structures. Only the Bonferroni test is evaluated.

In [1]:
import os
import sys
import time
from pathlib import Path
import math
import gudhi as gd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../../source"))
from TDA_Testing import *

# Simulation configuration
alpha = 0.05
B = 5000
sample_sizes = np.array([30, 100, 200, 400], dtype=int)  # n=m
max_n = int(sample_sizes.max())
master_seed = 20260923
circle_separation = 10.0
max_block_entries = 1_000_000
progress_every = 10

result_csv_path = Path("equal_intensity_bonferroni_type1.csv")
figure_path = Path("equal_intensity_bonferroni_type1.png")

## Persistence-diagram data generation

In [2]:
# ============================================================
# 1. Construct X(b,d) on a circle
# ============================================================

def circle_point_cloud(b, d, center):
    """
    Construct X(b,d) on the circle of radius d.

    Adjacent points in cyclic order have Euclidean distance 2b,
    except possibly for the last-first pair, whose distance is <= 2b.
    """
    if not (0 < b < d):
        raise ValueError("Require 0 < b < d.")

    theta = 2.0 * np.arcsin(b / d)
    N = int(np.ceil(2.0 * np.pi / theta))
    angles = theta * np.arange(N)

    cx, cy = center

    return np.column_stack([
        cx + d * np.cos(angles),
        cy + d * np.sin(angles)
    ])

def single_circle_diagram(b, d):
    X = circle_point_cloud(
        b=b,
        d=d,
        center=(0.0, 0.0)
    )
    D = cech_h1_diagram(X)

    if D.shape != (1, 2):
        raise RuntimeError(
            f"Expected exactly one H1 point, but obtained {len(D)}:\n{D}"
        )

    return D


# ============================================================
# 2. Compute Cech H1 persistence via Alpha complex
# ============================================================

def cech_h1_diagram(X):
    """
    Compute the H1 persistence diagram of the Cech filtration.

    AlphaComplex has the same persistent homology as the Euclidean
    Cech filtration. GUDHI stores squared radius values, so square
    roots are taken before returning the diagram.
    """
    alpha = gd.AlphaComplex(points=X)
    st = alpha.create_simplex_tree()
    st.compute_persistence()

    H1_squared = st.persistence_intervals_in_dimension(1)
    H1 = np.sqrt(H1_squared)

    persistence = H1[:, 1] - H1[:, 0]
    tol = 1e-10
    return H1[persistence > tol]


def sample_F(rng):
    """Sample (b,d) from the fixed distribution F on Omega."""
    b = rng.uniform(0.15, 0.25)
    d = rng.uniform(0.445, 0.55)
    return b, d

def generate_diagrams_P(n=100, seed=42):
    rng = np.random.default_rng(seed)
    diagrams = []

    for _ in range(n):
        b, d = sample_F(rng)

        D = single_circle_diagram(b, d)

        diagrams.append(np.vstack([D, D.copy()]))

    return diagrams


def generate_diagrams_Q(n=100, seed=123):
    rng = np.random.default_rng(seed)
    diagrams = []

    for _ in range(n):
        b1, d1 = sample_F(rng)
        b2, d2 = sample_F(rng)

        D1 = single_circle_diagram(b1, d1)
        D2 = single_circle_diagram(b2, d2)

        diagrams.append(np.vstack([D1, D2]))

    return diagrams

## Monte Carlo experiment

Each replication generates a fresh independent P sample and Q sample of size 400. The first 30, 100, 200, and 400 observations form nested samples within that replication. Different Monte Carlo replications do not reuse data. Every execution starts from replication 0; no checkpoint is loaded or saved.

In [3]:
func_weight = function_weight("Poly", poly_order=1)

In [5]:
rejections = np.zeros((len(sample_sizes), B), dtype=bool)
completed = 0
start_time = time.time()

# Reruns with the same master_seed reproduce the same Monte Carlo experiment.
seed_sequence = np.random.SeedSequence(master_seed)

replication_seeds = seed_sequence.generate_state(2 * B, dtype=np.uint64)

for replication in range(B):
    P_sample = generate_diagrams_P(
        n=max_n,
        seed=int(replication_seeds[2 * replication])
    )
    Q_sample = generate_diagrams_Q(
        n=max_n,
        seed=int(replication_seeds[2 * replication + 1])
    )

    if replication == 0:
        p_sizes, p_counts = np.unique(
            [len(D) for D in P_sample], return_counts=True
        )
        q_sizes, q_counts = np.unique(
            [len(D) for D in Q_sample], return_counts=True
        )
        print("P diagram sizes:", dict(zip(p_sizes, p_counts)))
        print("Q diagram sizes:", dict(zip(q_sizes, q_counts)))

    for size_index, n in enumerate(sample_sizes):
        result = bonferroni_test(
            P_sample[:n],
            Q_sample[:n],
            alpha=alpha,
            func_weight=func_weight,
            max_block_entries=max_block_entries
        )
        rejections[size_index, replication] = result["reject"]

    completed = replication + 1

    if completed % progress_every == 0 or completed == B:
        elapsed = time.time() - start_time
        current_rates = rejections[:, :completed].mean(axis=1)
        rate_text = ", ".join(
            f"n={n}: {rate:.4f}"
            for n, rate in zip(sample_sizes, current_rates)
        )
        print(
            f"{completed}/{B} completed | {rate_text} | "
            f"elapsed={elapsed / 60:.1f} min"
        )

P diagram sizes: {2: 400}
Q diagram sizes: {2: 400}
10/5000 completed | n=30: 0.0000, n=100: 0.0000, n=200: 0.0000, n=400: 0.0000 | elapsed=0.2 min
20/5000 completed | n=30: 0.0000, n=100: 0.0000, n=200: 0.0500, n=400: 0.0000 | elapsed=0.4 min
30/5000 completed | n=30: 0.0333, n=100: 0.0000, n=200: 0.0667, n=400: 0.0000 | elapsed=0.6 min
40/5000 completed | n=30: 0.0500, n=100: 0.0250, n=200: 0.0500, n=400: 0.0000 | elapsed=0.8 min
50/5000 completed | n=30: 0.0400, n=100: 0.0400, n=200: 0.0400, n=400: 0.0400 | elapsed=0.9 min
60/5000 completed | n=30: 0.0333, n=100: 0.0333, n=200: 0.0500, n=400: 0.0333 | elapsed=1.1 min
70/5000 completed | n=30: 0.0286, n=100: 0.0286, n=200: 0.0429, n=400: 0.0286 | elapsed=1.2 min
80/5000 completed | n=30: 0.0250, n=100: 0.0375, n=200: 0.0500, n=400: 0.0375 | elapsed=1.3 min
90/5000 completed | n=30: 0.0333, n=100: 0.0444, n=200: 0.0444, n=400: 0.0333 | elapsed=1.5 min
100/5000 completed | n=30: 0.0300, n=100: 0.0400, n=200: 0.0400, n=400: 0.0300 | ela

## Empirical type-I error

In [6]:
if completed != B:
    raise RuntimeError(f"Only {completed} of {B} replications are complete.")

empirical_type1 = rejections.mean(axis=1)
mcse = np.sqrt(empirical_type1 * (1.0 - empirical_type1) / B)
ci_lower = np.maximum(0.0, empirical_type1 - 1.96 * mcse)
ci_upper = np.minimum(1.0, empirical_type1 + 1.96 * mcse)

results = pd.DataFrame({
    "n": sample_sizes,
    "m": sample_sizes,
    "replications": B,
    "rejections": rejections.sum(axis=1),
    "empirical_type1": empirical_type1,
    "mcse": mcse,
    "ci_95_lower": ci_lower,
    "ci_95_upper": ci_upper
})

results.to_csv(result_csv_path, index=False)
results

,n,m,replications,rejections,empirical_type1,mcse,ci_95_lower,ci_95_upper
0,30,30,5000,121,0.0242,0.002173,0.019940,0.028460
1,100,100,5000,107,0.0214,0.002047,0.017389,0.025411
2,200,200,5000,113,0.0226,0.002102,0.018480,0.026720
3,400,400,5000,118,0.0236,0.002147,0.019392,0.027808
